# Pillar3d-v2.3-lam0.003: aux λ sweep on the decisive corpus (λ=0.003)

## Why
The gradient audit (`train_path_b --grad-audit`) showed λ=0.03 is **aux-dominated**: effective share
`λ·|g_aux|/|g_main| = 2.09` (the corrections drive training 2:1 over the base; gradients orthogonal,
cos≈0). pillar3b is converged → tiny `|g_main|≈0.23`; OOD crisis corrections → huge `|g_aux|≈15.8`.
So "λ=0.03 gentle" was an illusion, and v2.2's regression is largely a training-mechanics over-index.

**This run: corpus = decisive (v2.3 filter, 11,702) + gentler `--aux-lambda 0.003`**
→ predicted effective aux share ≈ **~0.21**. Everything else byte-identical to v2.1/v2.3 (warm-start
pillar3b_epoch_20, lr 5e-5, target-temp 0.5, 10 epochs). Corpus fixed across the λ sweep so this
isolates λ. Question: does a gentler aux **generalize / lift the floor better**, or do the gains need
the strong regime?

## Required Drive uploads (`MyDrive/alphatrain/`)
1. `colorlines_pillar3d_v2.tar.gz` — code (recipe flags unchanged)
2. **`corrections_corpus_mm05.pt`** — the decisive (v2.3 filter, 11,702) corpus
3. `v13_pillar3a.pt.gz`, `pillar3b_epoch_20.pt` — already on Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time
DRIVE = '/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True); os.makedirs('/content/crisis', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar3b_epoch_20.pt', '/content/alphatrain/data/pillar3b_epoch_20.pt')
shutil.copy(f'{DRIVE}/corrections_corpus_mm05.pt', '/content/crisis/corrections_corpus.pt')
import torch
_c = torch.load('/content/crisis/corrections_corpus.pt')
print('corpus anchors', _c['boards'].shape[0], _c['_stats'])
assert 9000 < _c['boards'].shape[0] < 20000 and _c['_stats']['min_margin'] >= 0.05, \
    "unexpected corpus size/min_margin — upload corrections_corpus_mm05.pt"
t0 = time.time()
!gzip -dc {DRIVE}/v13_pillar3a.pt.gz > /content/alphatrain/data/v13_pillar3a.pt
print(f"V13 tensor ({time.time()-t0:.0f}s)")
%cd /content
!pip install -q numpy numba scipy

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Train pillar3d-v2.3-lam0.003 (~12h) — only `--aux-lambda 0.003` differs

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/v13_pillar3a.pt \
    --amp --compile \
    --resume alphatrain/data/pillar3b_epoch_20.pt --warm-start \
    --epochs 10 --batch-size 32768 --lr 5e-5 --warmup-epochs 1 \
    --target-temperature 0.5 \
    --aux-corrections-corpus crisis/corrections_corpus.pt --aux-weighted \
    --aux-lambda 0.003 --aux-target-temperature 0.5 \
    --aux-holdout-frac 0.15 --aux-split-seed 0 \
    --aux-batch-size 256 --aux-warmup-epochs 2.0 \
    --aux-preflight-every 200 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar3d_v2_3_lam0.003_best.pt \
    --save-dir /content/checkpoints/pillar3d_v2_3_lam0.003 2>&1 | tee /content/pillar3d_v2_3_lam0.003_train.log

In [ ]:
!grep -E 'soft-preflight|soft-heldout|Train:|val:' /content/pillar3d_v2_3_lam0.003_train.log

In [ ]:
import glob, shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/pillar3d_v2_3_lam0.003/epoch_*.pt')):
    dst = f'{DRIVE}/pillar3d_v2_3_lam0.003_{os.path.basename(f)}'
    if not os.path.exists(dst):
        shutil.copy(f, dst); print('copied', os.path.basename(f))
shutil.copy('/content/pillar3d_v2_3_lam0.003_train.log', f'{DRIVE}/pillar3d_v2_3_lam0.003_train.log')

## Floor eval — 5,000 games, vs v2.1 (the bar) and the other λ
Baselines on 5,000 held-out (777000..781999):
- **v2.1-ep2 (bar):** mean **21,195**, P10 2,796, P50 15,121, <1000 2.6%, >10k 64%
- v2.2-ep2 (26.8k all, λ=0.03): mean 20,711, P10 2,552 (regressed)

Sweep epochs; with a gentler aux the policy moves less per epoch → best epoch may be LATER.

In [ ]:
%cd /content
for EP in [3, 5, 7, 10]:
    m = f'/content/checkpoints/pillar3d_v2_3_lam0.003/epoch_{EP}.pt'
    if not os.path.exists(m): continue
    print(f'===== v2.3-lam0.003 (λ=0.003) epoch {EP}  (5k held-out) =====')
    !python -m alphatrain.scripts.eval_parallel \
        --model {m} --policy-only \
        --seeds $(seq 777000 781999) --device cuda --workers 8 \
        2>&1 | grep -E 'P5|P10|mean|<1000|<500|>10000'

## Read it — λ=0.003 (effective aux share ≈ ~0.21)
- **Beats v2.1** → gentler aux generalizes better; the λ=0.03 regime was over-indexing. Pick the
  winning (λ, epoch); this becomes the recipe.
- **Ties v2.1** → λ not the limiter at this corpus; compare across the sweep (0.03 / 0.01 / 0.003).
- **Below v2.1** → too gentle, the gains need a stronger aux → label-quality / rollout-grounded teacher.
Compare the three λ side by side: if there's a monotone floor trend, that's the lever.